# Sim 6: Opus-Teacher PDA Distillation Sweep (Gemma-3-4b, RunPod)

**Teacher:** Claude Opus 4.6
**Student:** unsloth/gemma-3-4b-pt (4bit pre-quantized)
**Platform:** RunPod, any GPU with >=12GB VRAM

**Experiments:** Base vs CoT vs PDA-2/3/4/5 on GSM8K (200 questions)

**Setup:**
1. Pod: PyTorch 2.x template, one GPU (RTX 3090/4090/5000-Ada/A100 all fine)
2. Upload to `/workspace/`:
   - opus_cot_gsm8k.jsonl
   - opus_pda2_gsm8k.jsonl
   - opus_pda3_gsm8k.jsonl
   - opus_pda4_gsm8k.jsonl
   - opus_pda5_gsm8k.jsonl
3. Run All


In [ ]:
%%capture
!pip install unsloth
!pip install --no-deps trl peft accelerate bitsandbytes
!pip install datasets

## 1. Load Training Data

In [ ]:
import os, json, re, random

# Kill env-vars that trigger accelerate distributed mode on multi-GPU pods
for k in ["WORLD_SIZE", "RANK", "LOCAL_RANK", "MASTER_ADDR", "MASTER_PORT",
         "ACCELERATE_USE_FSDP", "ACCELERATE_USE_DEEPSPEED", "ACCELERATE_MIXED_PRECISION"]:
    os.environ.pop(k, None)
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["ACCELERATE_BYPASS_DEVICE_MAP"] = "true"

DATA_DIR = "/workspace"
OUTPUT_DIR = "/workspace"

def load_correct(filename, reasoning_key):
    path = os.path.join(DATA_DIR, filename)
    examples = []
    with open(path) as f:
        for line in f:
            d = json.loads(line)
            if d.get("correct", False):
                examples.append({
                    "question": d["question"],
                    "reasoning": d.get(reasoning_key, d.get("reasoning", "")),
                })
    return examples

data = {
    "cot":  load_correct("opus_cot_gsm8k.jsonl",  "reasoning"),
    "pda2": load_correct("opus_pda2_gsm8k.jsonl", "pda_reasoning"),
    "pda3": load_correct("opus_pda3_gsm8k.jsonl", "pda_reasoning"),
    "pda4": load_correct("opus_pda4_gsm8k.jsonl", "pda_reasoning"),
    "pda5": load_correct("opus_pda5_gsm8k.jsonl", "pda_reasoning"),
}

for k, v in data.items():
    print(f"{k}: {len(v)} correct examples")

## 2. Setup

In [ ]:
import gc, torch
from unsloth import FastModel
from trl import SFTTrainer
from transformers import TrainingArguments
from datasets import Dataset

max_seq_length = 2048
MODEL_NAME = "unsloth/gemma-3-4b-pt"

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f}GB")
print(f"bf16 supported: {torch.cuda.is_bf16_supported()}")

def make_dataset(examples):
    texts = [f"Question: {ex['question']}\n\nSolution: {ex['reasoning']}"
             for ex in examples]
    return Dataset.from_list([{"text": t} for t in texts])

def train_adapter(variant_name, examples):
    save_path = os.path.join(OUTPUT_DIR, f"adapter-opus-{variant_name}")
    if os.path.exists(os.path.join(save_path, "adapter_model.safetensors")):
        print(f"=== {variant_name}: already trained, skipping ===\n")
        return

    print(f"=== Training {variant_name} on {len(examples)} examples ===")

    model, tokenizer = FastModel.from_pretrained(
        model_name=MODEL_NAME,
        max_seq_length=max_seq_length,
        dtype=None,  # auto: bf16 on Ada/Ampere
        load_in_4bit=True,
        full_finetuning=False,
    )
    model = FastModel.get_peft_model(
        model, r=16,
        target_modules=["q_proj","k_proj","v_proj","o_proj",
                        "gate_proj","up_proj","down_proj"],
        lora_alpha=16, lora_dropout=0, bias="none",
        use_gradient_checkpointing="unsloth", random_state=42,
        finetune_vision_layers=False,
        finetune_language_layers=True,
        finetune_attention_modules=True,
        finetune_mlp_modules=True,
    )

    dataset = make_dataset(examples)

    trainer = SFTTrainer(
        model=model, tokenizer=tokenizer,
        train_dataset=dataset, dataset_text_field="text",
        max_seq_length=max_seq_length, dataset_num_proc=2, packing=False,
        args=TrainingArguments(
            per_device_train_batch_size=2, gradient_accumulation_steps=4,
            warmup_steps=10, num_train_epochs=3, learning_rate=2e-4,
            fp16=not torch.cuda.is_bf16_supported(),
            bf16=torch.cuda.is_bf16_supported(),
            logging_steps=10, optim="adamw_8bit",
            weight_decay=0.01, lr_scheduler_type="linear",
            seed=42, output_dir=save_path + "-checkpoints",
            save_strategy="no", report_to="none",
            ddp_find_unused_parameters=False,
        ),
    )
    stats = trainer.train()
    print(f"Loss: {stats.training_loss:.4f}")

    model.save_pretrained(save_path)
    tokenizer.save_pretrained(save_path)
    print(f"Saved to {save_path}")

    del model, tokenizer, trainer, dataset
    gc.collect()
    torch.cuda.empty_cache()
    print(f"VRAM after cleanup: {torch.cuda.memory_allocated()/1e9:.2f}GB\n")

print("Setup done.")

## 3. Train all variants

In [ ]:
for variant, examples in data.items():
    train_adapter(variant, examples)

print("=== All adapters trained ===")

## 4. Evaluation on GSM8K

In [ ]:
from datasets import load_dataset
from peft import PeftModel

N_EVAL = 200

def extract_number(text):
    m = re.search(r'####\s*(-?[\d,]+\.?\d*)', text)
    if m: return float(m.group(1).replace(",", ""))
    nums = re.findall(r'-?[\d,]+\.?\d*', text)
    for n in reversed(nums):
        c = n.replace(",", "").strip()
        if c and c != "-":
            try: return float(c)
            except: continue
    return None

def normalize(s):
    if s is None: return None
    s = str(s).strip().replace(" ", "").lower()
    try: return str(float(s))
    except: return s

gsm8k_test = load_dataset("openai/gsm8k", "main", split="test")
random.seed(42)
test_idx = list(range(len(gsm8k_test)))
random.shuffle(test_idx)
test_idx = test_idx[:N_EVAL]

def evaluate(model, tokenizer, tag):
    correct = total = 0
    for i, idx in enumerate(test_idx):
        item = gsm8k_test[idx]
        m = re.search(r'####\s*(-?[\d,]+\.?\d*)', item["answer"])
        if not m: continue
        gt = float(m.group(1).replace(",", ""))

        # trailing space — matches training format
        prompt = f"Question: {item['question']}\n\nSolution: "
        ids = tokenizer(prompt, return_tensors="pt", add_special_tokens=True).input_ids.to(model.device)

        with torch.inference_mode():
            out = model.generate(
                input_ids=ids,
                max_new_tokens=300,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
            )
        resp = tokenizer.decode(out[0][ids.shape[-1]:], skip_special_tokens=True)
        pred = extract_number(resp)

        if pred is not None and normalize(str(pred)) == normalize(str(gt)):
            correct += 1
        total += 1

        if (i+1) % 50 == 0:
            print(f"  [{tag}] {i+1}/{N_EVAL}: {correct}/{total} ({100*correct/total:.1f}%)")

    acc = round(100*correct/total, 1) if total else 0
    print(f"  [{tag}] Final: {correct}/{total} ({acc}%)")
    return {"correct": correct, "total": total, "accuracy": acc}

print(f"Ready. Test set: {len(test_idx)} questions")

In [ ]:
# Load base model fresh for eval
model, tokenizer = FastModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=max_seq_length,
    dtype=None,
    load_in_4bit=True,
    full_finetuning=False,
)
FastModel.for_inference(model)

# Sanity check on raw base model (no adapter loaded yet)
_ids = tokenizer("The capital of France is", return_tensors="pt").input_ids.to(model.device)
with torch.inference_mode():
    _out = model.generate(input_ids=_ids, max_new_tokens=20, do_sample=False,
                          pad_token_id=tokenizer.eos_token_id)
_str = tokenizer.decode(_out[0][_ids.shape[-1]:], skip_special_tokens=True)
print(f"SANITY CHECK -> {_str!r}")
assert "Paris" in _str, f"Base model broken: {_str!r}"
print("Model OK.\n")

# Baseline eval (no adapter)
results = {}
print("="*60)
print("BASELINE (no adapter)")
print("="*60)
results["base"] = evaluate(model, tokenizer, "Base")

In [ ]:
# Free base model — fresh load per adapter via Unsloth's native loader
del model
gc.collect()
torch.cuda.empty_cache()

# Unsloth loads base+LoRA jointly via fast-path when adapter_path is passed as model_name.
# Do NOT use PeftModel.from_pretrained — that bypasses the fast-path (CPU-fallback, 2 tok/s).
for variant in ["cot", "pda2", "pda3", "pda4", "pda5"]:
    adapter_path = os.path.join(OUTPUT_DIR, f"adapter-opus-{variant}")
    if not os.path.exists(adapter_path):
        print(f"Skipping {variant} (adapter not found)")
        continue

    print(f"\n{'='*60}")
    print(f"{variant.upper()} DISTILLED")
    print(f"{'='*60}")

    m, tok = FastModel.from_pretrained(
        model_name=adapter_path,     # adapter path, not base model
        max_seq_length=max_seq_length,
        dtype=None,
        load_in_4bit=True,
        full_finetuning=False,
    )
    # No for_inference(), no PeftModel wrap — Unsloth handles it

    results[variant] = evaluate(m, tok, variant.upper())

    del m, tok
    gc.collect()
    torch.cuda.empty_cache()
    print(f"VRAM after cleanup: {torch.cuda.memory_allocated()/1e9:.2f}GB")

## 5. Results

In [ ]:
print("\n" + "=" * 70)
print("  SIM 6: OPUS-TEACHER PDA DISTILLATION + PERSPECTIVE SWEEP")
print("=" * 70)
print(f"  Teacher: Claude Opus 4.6")
print(f"  Student: {MODEL_NAME}")
print(f"  Benchmark: GSM8K ({N_EVAL} questions)")
print("=" * 70)

base_acc = results.get("base", {}).get("accuracy", 0)
cot_acc = results.get("cot", {}).get("accuracy", 0)

print(f"\n  {'Variant':<12} {'Accuracy':>10} {'vs Base':>10} {'vs CoT':>10}")
print("  " + "-" * 42)

best_variant = "base"
best_acc = base_acc

for variant in ["base", "cot", "pda2", "pda3", "pda4", "pda5"]:
    if variant not in results:
        continue
    acc = results[variant]["accuracy"]
    vs_base = acc - base_acc
    vs_cot = acc - cot_acc if variant != "base" else 0

    base_str = f"+{vs_base:.1f}pp" if vs_base > 0 else f"{vs_base:.1f}pp" if vs_base < 0 else "---"
    cot_str = f"+{vs_cot:.1f}pp" if vs_cot > 0 else f"{vs_cot:.1f}pp" if vs_cot < 0 else "---"
    marker = " <-- best" if acc > best_acc else ""
    if acc > best_acc:
        best_acc = acc
        best_variant = variant

    print(f"  {variant:<12} {acc:>8.1f}%  {base_str:>10} {cot_str:>10}{marker}")

print("  " + "-" * 42)
print(f"\n  Best variant: {best_variant} ({best_acc}%)")

pda_accs = {k: v["accuracy"] for k, v in results.items() if k.startswith("pda")}
if pda_accs and cot_acc > 0:
    best_pda = max(pda_accs, key=pda_accs.get)
    print(f"  Best PDA: {best_pda} ({pda_accs[best_pda]}%)")
    all_beat = all(v > cot_acc for v in pda_accs.values())
    any_beat = any(v > cot_acc for v in pda_accs.values())
    if all_beat:
        print("  All PDA variants outperform CoT distillation.")
    elif any_beat:
        print("  Some PDA variants outperform CoT distillation.")
    else:
        print("  No PDA variant outperforms CoT distillation.")

with open(os.path.join(OUTPUT_DIR, "sim6_results.json"), "w") as f:
    json.dump(results, f, indent=2)
print(f"\n  Saved to {OUTPUT_DIR}/sim6_results.json")